In [1]:
import os

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\babyp\Project-3-Retail-Demand-Forecasting-Inventory-Optimization\notebooks


In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
APP_PATH = PROJECT_DIR / "app.py"

print("Project folder:", PROJECT_DIR)
print("App file path:", APP_PATH)

Project folder: c:\Users\babyp\Project-3-Retail-Demand-Forecasting-Inventory-Optimization
App file path: c:\Users\babyp\Project-3-Retail-Demand-Forecasting-Inventory-Optimization\app.py


In [3]:

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
APP_PATH = PROJECT_DIR / "app.py"

if APP_PATH.exists():
    print(f"STOP: {APP_PATH} already exists.")
    print("No file was changed. Check the existing app.py before replacing it.")
else:
    app_code = r'''
import streamlit as st
import pandas as pd
import plotly.express as px
from google.cloud import bigquery

PROJECT_ID = "sincere-quasar-511010-v3"
DATASET = "retail_demand_raw"

st.set_page_config(
    page_title="Retail Demand Dashboard",
    page_icon="📊",
    layout="wide",
)

st.title("Retail Demand Forecasting & Inventory Optimization")
st.caption("Week 4 | Streamlit, BigQuery, LightGBM and Prophet")

@st.cache_resource
def get_client():
    return bigquery.Client(project=PROJECT_ID)

@st.cache_data(ttl=300)
def load_forecasts():
    client = get_client()

    lgb_query = f"""
        SELECT item_id, store_id, forecast_date,
               actual_units, predicted_units, mae
        FROM `{PROJECT_ID}.{DATASET}.forecast_results`
        WHERE model_name = 'LightGBM'
    """

    prophet_query = f"""
        SELECT forecast_date, predicted_units,
               lower_bound, upper_bound
        FROM `{PROJECT_ID}.{DATASET}.prophet_forecast_results`
    """

    lgb = client.query(lgb_query).to_dataframe()
    prophet = client.query(prophet_query).to_dataframe()

    for df in (lgb, prophet):
        df["forecast_date"] = pd.to_datetime(df["forecast_date"])

    lgb["category_group"] = lgb["item_id"].str.split("_").str[0]
    return lgb, prophet

try:
    lgb, prophet = load_forecasts()
except Exception as exc:
    st.error("Unable to load BigQuery forecasts.")
    st.info(
        "Check Google Cloud authentication, project permissions, "
        "and that both forecast tables exist."
    )
    st.exception(exc)
    st.stop()

if lgb.empty or prophet.empty:
    st.warning("One or both forecast tables contain no rows.")
    st.stop()

st.sidebar.header("Filters")
stores = sorted(lgb["store_id"].dropna().unique().tolist())
store = st.sidebar.selectbox("Store", stores)

filtered = lgb[lgb["store_id"] == store].copy()
categories = sorted(
    filtered["category_group"].dropna().unique().tolist()
)
category = st.sidebar.selectbox("Product group", ["All"] + categories)

if category != "All":
    filtered = filtered[filtered["category_group"] == category]

a, b, c = st.columns(3)
a.metric("Available stores", int(lgb["store_id"].nunique()))
b.metric("Filtered prediction rows", f"{len(filtered):,}")
c.metric(
    "Recorded MAE",
    f'{filtered["mae"].mean():,.2f}' if not filtered.empty else "N/A",
)

st.subheader("1. LightGBM: Actual vs Predicted Demand")

if filtered.empty:
    st.info("No records match the selected filters.")
else:
    daily = (
        filtered.groupby("forecast_date", as_index=False)
        .agg(
            Actual=("actual_units", "sum"),
            Predicted=("predicted_units", "sum"),
        )
        .sort_values("forecast_date")
    )

    plot_data = daily.melt(
        id_vars="forecast_date",
        var_name="Measure",
        value_name="Units",
    )

    fig = px.line(
        plot_data,
        x="forecast_date",
        y="Units",
        color="Measure",
        markers=True,
    )
    st.plotly_chart(fig, use_container_width=True)
    st.dataframe(daily, use_container_width=True)

st.subheader("2. Prophet Aggregate Forecast")

st.caption(
    "Showing the latest 30 dates available in the Prophet table. "
    "These are not necessarily future dates."
)
p30 = prophet.sort_values("forecast_date").tail(30)

fig2 = px.line(
    p30,
    x="forecast_date",
    y="predicted_units",
    title="Aggregate predicted demand",
)
fig2.add_scatter(
    x=p30["forecast_date"],
    y=p30["lower_bound"],
    mode="lines",
    name="Lower bound",
)
fig2.add_scatter(
    x=p30["forecast_date"],
    y=p30["upper_bound"],
    mode="lines",
    name="Upper bound",
)
st.plotly_chart(fig2, use_container_width=True)

st.subheader("3. What-if Scenario: 10% Price Reduction")
st.write(
    "Explore a hypothetical demand response. This is an assumption-based "
    "scenario, not a trained price-elasticity model."
)

elasticity = st.slider(
    "Assumed price elasticity",
    min_value=-3.0,
    max_value=0.0,
    value=-1.0,
    step=0.1,
)

baseline = float(filtered["predicted_units"].sum()) if not filtered.empty else 0
price_change = -0.10
change = (1 + price_change) ** elasticity - 1
scenario = baseline * (1 + change)

x, y, z = st.columns(3)
x.metric("Baseline predicted units", f"{baseline:,.1f}")
y.metric("Scenario units", f"{scenario:,.1f}")
z.metric("Assumed demand change", f"{change:.1%}")

st.subheader("4. Download Filtered Results")
st.download_button(
    "Download CSV",
    data=filtered.to_csv(index=False).encode("utf-8"),
    file_name="filtered_lightgbm_forecasts.csv",
    mime="text/csv",
)

st.caption(
    "LightGBM records currently stored are test predictions. "
    "They should not be presented as a verified 30-day future forecast."
)
'''

    APP_PATH.write_text(app_code.strip() + "\n", encoding="utf-8")
    print("Created:", APP_PATH)
    print("Next: install requirements and launch Streamlit.")


STOP: c:\Users\babyp\Project-3-Retail-Demand-Forecasting-Inventory-Optimization\app.py already exists.
No file was changed. Check the existing app.py before replacing it.


In [4]:

import subprocess
import sys

packages = [
    "streamlit",
    "plotly",
    "google-cloud-bigquery",
    "pandas",
    "db-dtypes",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
print("Dashboard dependencies installed.")


Dashboard dependencies installed.


In [5]:

from pathlib import Path

requirements = """streamlit
plotly
google-cloud-bigquery
pandas
db-dtypes
"""

(PROJECT_DIR / "requirements.txt").write_text(
    requirements, encoding="utf-8"
)

print("Created:", PROJECT_DIR / "requirements.txt")


Created: c:\Users\babyp\Project-3-Retail-Demand-Forecasting-Inventory-Optimization\requirements.txt


In [6]:

import subprocess
import sys

streamlit_command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_PATH),
]

subprocess.Popen(streamlit_command, cwd=str(PROJECT_DIR))
print("Streamlit launch command sent.")
print("Open http://localhost:8501 in your browser.")


Streamlit launch command sent.
Open http://localhost:8501 in your browser.


In [7]:
import requests
import time

url = "http://localhost:8501"

try:
    response = requests.get(url, timeout=10)
    print("HTTP status:", response.status_code)

    if response.status_code == 200:
        print("Streamlit server is responding.")
    else:
        print("The server responded, but check the app in your browser.")
except Exception as e:
    print("Could not connect:", e)
    print("Check the terminal where Streamlit was launched.")

Could not connect: HTTPConnectionPool(host='localhost', port=8501): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8501): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
Check the terminal where Streamlit was launched.


In [1]:

import time
from google.cloud import bigquery

print("Step 1: Import successful")

start = time.time()
client = bigquery.Client(project="sincere-quasar-511010-v3")
print(f"Step 2: BigQuery client created in {time.time() - start:.2f} seconds")

print("Step 3: Testing a tiny query...")
start = time.time()

job = client.query("SELECT 1 AS test_value")
result = job.result(timeout=30)

print("Step 4: Query result:", list(result))
print(f"Step 5: Finished in {time.time() - start:.2f} seconds")


Step 1: Import successful
Step 2: BigQuery client created in 1.88 seconds
Step 3: Testing a tiny query...
Step 4: Query result: [Row((1,), {'test_value': 0})]
Step 5: Finished in 2.51 seconds


In [2]:

from google.cloud import bigquery

client = bigquery.Client(project="sincere-quasar-511010-v3")

query = """
SELECT forecast_date, predicted_units
FROM `sincere-quasar-511010-v3.retail_demand_raw.prophet_forecast_results`
LIMIT 5
"""

job = client.query(query)
rows = job.result(timeout=30)

for row in rows:
    print(row.forecast_date, row.predicted_units)


2011-01-29 171330.71509912354
2011-01-30 171916.8795041895
2011-01-31 172494.59381876196
2011-02-01 173064.3262297975
2011-02-02 173626.16032838624


In [3]:

from google.cloud import bigquery

client = bigquery.Client(project="sincere-quasar-511010-v3")

query = """
SELECT forecast_date
FROM `sincere-quasar-511010-v3.retail_demand_raw.prophet_forecast_results`
ORDER BY forecast_date DESC
LIMIT 1
"""

rows = list(client.query(query).result(timeout=30))
print("Latest Prophet forecast date:", rows[0].forecast_date if rows else "No rows found")


Latest Prophet forecast date: 2016-06-12
